# Objective

* Transfer X,Y(NAD 1983 StatePlane New York West/EPSG:2262) to longitude and latitude.

*  Use Folium to plot points on a map of Rochester, NY, to help verify if they are in the correct location.

### Library

In [1]:
pip install pandas geopandas pyproj folium

In [2]:
import os
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
import folium
from folium.plugins import MarkerCluster, HeatMap

### Set Up


In [3]:
INPUT_CSV = "RPD_-_Part_I_Crime_2011_to_Present.csv"
X_Col = "X"
Y_Col = "Y"
SOURCE_EPSG = 2262
OUTPUT_CSV = "output_with_latlon.csv"
POINT_MAP_HTML = "rochester_points.html"
HEATMAP_HTML = "rochester_heatmap.html"

### Code

In [15]:
ROC_CENTER = [43.1566, -77.6088]
INIT_ZOOM = 11
# =========================

# 1) Read Dataset
df = pd.read_csv(INPUT_CSV)

# Drop Missing value
df = df.dropna(subset=[X_Col, Y_Col]).copy()

# 2) Construct GeoDataFrame(Original Projection: EPSG:2262)
gdf = gpd.GeoDataFrame(
    df,
    geometry=gpd.points_from_xy(df[X_Col], df[Y_Col]),
    crs=f"EPSG:{SOURCE_EPSG}"
)

# 3) Transfer（EPSG:4326）
gdf_wgs = gdf.to_crs(epsg=4326)

# 4) Write （longitude / latitude）
gdf_wgs["longitude"] = gdf_wgs.geometry.x
gdf_wgs["latitude"]  = gdf_wgs.geometry.y

# 5) Reorder columns and save New CSV file with longitude / latitude
cols = gdf_wgs.columns.tolist()
# Remove longitude and latitude from their current positions
cols.remove('longitude')
cols.remove('latitude')

# Insert longitude and latitude at the desired positions (index 2 and 3)
# Note: Inserting at index 2 will put longitude at index 2, and then
# inserting at index 3 will put latitude at index 3 (shifting subsequent columns).
# So we insert latitude first, then longitude.
cols.insert(2, 'longitude')
cols.insert(3, 'latitude')

# Select columns in the new order
gdf_wgs_reordered = gdf_wgs[cols]

# Drop the geometry column before saving to CSV
gdf_wgs_reordered = gdf_wgs_reordered.drop(columns=['geometry'])


# Save the reordered DataFrame to CSV
gdf_wgs_reordered.to_csv(OUTPUT_CSV, index=False)
print(f"[OK] :{OUTPUT_CSV}")

[OK] :output_with_latlon.csv


In [16]:
# 6) Simple range check (approximate latitude and longitude range near Rochester)
lon_min, lon_max = -78.2, -77.2
lat_min, lat_max =  42.9,  43.4
lon_ok = gdf_wgs["longitude"].between(lon_min, lon_max).mean()
lat_ok = gdf_wgs["latitude"].between(lat_min, lat_max).mean()
print(f"[Check] Proportion of longitude within Rochester's range:{lon_ok:.2%}，Latitude proportion:{lat_ok:.2%}")

# 7) Draw an interactive point map (zoom in/out)
m_points = folium.Map(location=ROC_CENTER, zoom_start=INIT_ZOOM, tiles="OpenStreetMap")
cluster = MarkerCluster().add_to(m_points)

for row in gdf_wgs[["latitude", "longitude"]].itertuples(index=False):

    folium.CircleMarker(
        location=(row.latitude, row.longitude),
        radius=2,
        weight=1,
        opacity=0.8,
        fill=True,
        fill_opacity=0.7
    ).add_to(cluster)

m_points.save(POINT_MAP_HTML)
print(f"[OK] Points saved:{POINT_MAP_HTML}")

# 8) Draw a heatmap ）
m_heat = folium.Map(location=ROC_CENTER, zoom_start=INIT_ZOOM, tiles="OpenStreetMap")
heat_data = gdf_wgs[["latitude", "longitude"]].dropna().values.tolist()
HeatMap(heat_data, radius=10, blur=15, max_zoom=13).add_to(m_heat)
m_heat.save(HEATMAP_HTML)
print(f"[OK] Heat Map saved:{HEATMAP_HTML}")

[Check] Proportion of longitude within Rochester's range:100.00%，Latitude proportion:100.00%
[OK] Points saved:rochester_points.html
[OK] Heat Map saved:rochester_heatmap.html


In [17]:
# Read the original CSV file
original_df = pd.read_csv(INPUT_CSV)

# Read the new CSV file
new_df = pd.read_csv(OUTPUT_CSV)

# Print the dimensions of both dataframes
print(f"Shape of original dataframe ({INPUT_CSV}): {original_df.shape}")
print(f"Shape of new dataframe ({OUTPUT_CSV}): {new_df.shape}")

Shape of original dataframe (RPD_-_Part_I_Crime_2011_to_Present.csv): (139186, 40)
Shape of new dataframe (output_with_latlon.csv): (138535, 42)
